In [1]:
import sys
print(sys.executable)

D:\BERT_Project\bert_env\Scripts\python.exe


In [2]:
# Cell 2: Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"Torch file: {torch.__file__}")

PyTorch version: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA GeForce RTX 3070
GPU memory: 8.59 GB
Torch file: D:\BERT_Project\bert_env\lib\site-packages\torch\__init__.py


In [3]:
# Cell 3: Import libraries
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score, accuracy_score
import pickle
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')
from collections import Counter
import time
from datetime import datetime

D:\BERT_Project\bert_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Cell 4: Load data
train_df = pd.read_csv("train/train.csv")
dev_df = pd.read_csv("dev/dev.csv")
test_df = pd.read_csv("test/test.csv")

print(f"Train shape: {train_df.shape}")
print(f"Dev shape: {dev_df.shape}")
print(f"Test shape: {test_df.shape}")
train_df.head()

Train shape: (8000, 15)
Dev shape: (1600, 15)
Test shape: (1600, 15)


,original_id,Complain_date,complaint_title,text_en,location_address,category_primary,category_secondary,resolution_status,comment_count,Hindi,Hinglish,severity,severity_score,high_keywords_matched,medium_keywords_matched
0,BBMP_008791,03-08-2020 13:41,Streetlights not working,Diagonal road connecting bendre circle and BBM...,Opposite govt nature cure hospital,Electricity,Streetlights & Public Lighting,Open,0,बेंद्रे सर्कल और बीबीएमपी योग हॉल को जोड़ने वा...,Bendre circle aur BBMP yoga hall ko jodnewali ...,MEDIUM,4,NaN,not working
1,BBMP_004679,09-06-2019 07:21,This is in middle of the road,This is in middle of the road,NaN,Road & Transportation,Road Damage & Potholes,Resolved,2,यह सड़क के बीच में है।,Yeh sadak ke beech mein hai.,LOW,0,NaN,NaN
2,BBMP_006119,10/31/2019 17:36,There is big hole on the road where people wil...,There is big hole on the road where people wil...,NaN,Road & Transportation,Road Damage & Potholes,Resolved,1,सड़क पर एक बड़ा गड्ढा है जहाँ लोग बहुत तेज़ी स...,Sadak par ek bada gaddha hai jahan log bahut t...,MEDIUM,3,NaN,issue
3,BBMP_000306,1/22/2019 19:00,A lot of garbage is accumulated on harlur road...,A lot of garbage is accumulated on harlur road...,NaN,Waste Management,Garbage Dumping & Black Spots,Re-opened,4,हरलूर रोड पर जलाशय के पास बहुत कचरा जमा हो गया...,Harlur Road par jalashay ke paas bahut kachra ...,LOW,0,NaN,NaN
4,BBMP_008274,2/15/2020 16:35,Road clearing material is just left. This plac...,Road clearing material is just left. This plac...,NaN,Waste Management,Garbage Dumping & Black Spots,Resolved,1,सड़क साफ करने वाली सामग्री बस छोड़ दी गई है। इ...,Sadak saaf karne wali samagri bas chhod di gay...,LOW,0,NaN,NaN


In [5]:
# Cell 5: Preprocess data (unchanged from original)
def preprocess(df):
    df["text"] = (
        df["complaint_title"].astype(str) + " " +
        df["text_en"].astype(str) + " " +
        df["Hindi"].astype(str) + " " +
        df["Hinglish"].astype(str)
    )
    
    df = df.dropna(subset=[
        "text", "category_primary",
        "category_secondary", "severity"
    ])
    
    return df.reset_index(drop=True)

train_df = preprocess(train_df)
dev_df = preprocess(dev_df)
test_df = preprocess(test_df)

train_df.head()

,original_id,Complain_date,complaint_title,text_en,location_address,category_primary,category_secondary,resolution_status,comment_count,Hindi,Hinglish,severity,severity_score,high_keywords_matched,medium_keywords_matched,text
0,BBMP_008791,03-08-2020 13:41,Streetlights not working,Diagonal road connecting bendre circle and BBM...,Opposite govt nature cure hospital,Electricity,Streetlights & Public Lighting,Open,0,बेंद्रे सर्कल और बीबीएमपी योग हॉल को जोड़ने वा...,Bendre circle aur BBMP yoga hall ko jodnewali ...,MEDIUM,4,NaN,not working,Streetlights not working Diagonal road connect...
1,BBMP_004679,09-06-2019 07:21,This is in middle of the road,This is in middle of the road,NaN,Road & Transportation,Road Damage & Potholes,Resolved,2,यह सड़क के बीच में है।,Yeh sadak ke beech mein hai.,LOW,0,NaN,NaN,This is in middle of the road This is in middl...
2,BBMP_006119,10/31/2019 17:36,There is big hole on the road where people wil...,There is big hole on the road where people wil...,NaN,Road & Transportation,Road Damage & Potholes,Resolved,1,सड़क पर एक बड़ा गड्ढा है जहाँ लोग बहुत तेज़ी स...,Sadak par ek bada gaddha hai jahan log bahut t...,MEDIUM,3,NaN,issue,There is big hole on the road where people wil...
3,BBMP_000306,1/22/2019 19:00,A lot of garbage is accumulated on harlur road...,A lot of garbage is accumulated on harlur road...,NaN,Waste Management,Garbage Dumping & Black Spots,Re-opened,4,हरलूर रोड पर जलाशय के पास बहुत कचरा जमा हो गया...,Harlur Road par jalashay ke paas bahut kachra ...,LOW,0,NaN,NaN,A lot of garbage is accumulated on harlur road...
4,BBMP_008274,2/15/2020 16:35,Road clearing material is just left. This plac...,Road clearing material is just left. This plac...,NaN,Waste Management,Garbage Dumping & Black Spots,Resolved,1,सड़क साफ करने वाली सामग्री बस छोड़ दी गई है। इ...,Sadak saaf karne wali samagri bas chhod di gay...,LOW,0,NaN,NaN,Road clearing material is just left. This plac...


In [6]:
# Cell 6: Encode labels (same as original, just added class distribution info)
le_primary = LabelEncoder()
le_subcat = LabelEncoder()
le_severity = LabelEncoder()

train_df["primary_label"] = le_primary.fit_transform(train_df["category_primary"])
train_df["subcat_label"] = le_subcat.fit_transform(train_df["category_secondary"])
train_df["severity_label"] = le_severity.fit_transform(train_df["severity"])

dev_df["primary_label"] = le_primary.transform(dev_df["category_primary"])
dev_df["subcat_label"] = le_subcat.transform(dev_df["category_secondary"])
dev_df["severity_label"] = le_severity.transform(dev_df["severity"])

# Print class distribution for awareness
print("Primary categories:", len(le_primary.classes_))
print("Subcategories:", len(le_subcat.classes_))
print("Severity levels:", len(le_severity.classes_))
print("\nSeverity distribution:")
print(Counter(train_df["severity_label"]))

Primary categories: 8
Subcategories: 30
Severity levels: 3

Severity distribution:
Counter({1: 5261, 2: 1553, 0: 977})


In [7]:
# Cell 7: Load tokenizer
model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
print(f"Tokenizer loaded: {model_name}")

Tokenizer loaded: xlm-roberta-base


In [8]:
# Cell 8: Dataset class (same as original, optimized)
class ComplaintDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.texts = df["text"].tolist()
        self.primary = df["primary_label"].tolist()
        self.subcat = df["subcat_label"].tolist()
        self.severity = df["severity_label"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "primary": torch.tensor(self.primary[idx], dtype=torch.long),
            "subcat": torch.tensor(self.subcat[idx], dtype=torch.long),
            "severity": torch.tensor(self.severity[idx], dtype=torch.long)
        }

In [9]:
# Cell 9: Create dataloaders (back to original, no sampling)
train_dataset = ComplaintDataset(train_df, tokenizer, max_length=128)
dev_dataset = ComplaintDataset(dev_df, tokenizer, max_length=128)

batch_size = 8  # Keep same as original

train_loader = DataLoader(
    train_dataset, 
    batch_size=batch_size, 
    shuffle=True,
    num_workers=0,
    pin_memory=True if torch.cuda.is_available() else False
)

dev_loader = DataLoader(
    dev_dataset, 
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=True if torch.cuda.is_available() else False
)

print(f"Train batches: {len(train_loader)}")
print(f"Dev batches: {len(dev_loader)}")

Train batches: 974
Dev batches: 195


In [10]:
# Cell 10: Model - Keep original architecture but with minor improvements
class MultiTaskModel(nn.Module):
    def __init__(self, model_name, n1, n2, n3):
        super().__init__()
        self.base = AutoModel.from_pretrained(model_name)
        hidden = self.base.config.hidden_size
        
        # Keep original dropout rate (0.3) but only add dropout, no extra layers
        self.dropout = nn.Dropout(0.3)
        
        # Task-specific heads (no extra complexity)
        self.primary_head = nn.Linear(hidden, n1)
        self.subcat_head = nn.Linear(hidden, n2)
        self.severity_head = nn.Linear(hidden, n3)
    
    def forward(self, input_ids, attention_mask):
        outputs = self.base(input_ids=input_ids, attention_mask=attention_mask)
        cls = outputs.last_hidden_state[:, 0, :]
        x = self.dropout(cls)
        
        return (
            self.primary_head(x),
            self.subcat_head(x),
            self.severity_head(x)
        )

In [11]:
# Cell 11: Initialize model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if torch.cuda.is_available():
    torch.cuda.empty_cache()

model = MultiTaskModel(
    model_name,
    len(le_primary.classes_),
    len(le_subcat.classes_),
    len(le_severity.classes_)
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

Using device: cuda
Total parameters: 278,075,177


In [12]:
# Cell 12: Setup optimizer, scheduler, and loss functions
# Use standard CrossEntropyLoss (no class weights by default, or optional)
criterion = nn.CrossEntropyLoss()

# Optimizer (same as original)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=0.01  # Keep weight decay for regularization
)

# Scheduler with warmup (helps with convergence)
num_epochs = 5
num_training_steps = len(train_loader) * num_epochs
num_warmup_steps = int(0.1 * num_training_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

print(f"Training steps: {num_training_steps}")
print(f"Warmup steps: {num_warmup_steps}")

Training steps: 4870
Warmup steps: 487


In [13]:
# Cell 13: Training function with progress monitoring
def train_epoch(model, dataloader, optimizer, scheduler, epoch, num_epochs):
    model.train()
    total_loss = 0
    all_predictions = []
    all_labels = []
    
    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch}/{num_epochs} [Training]")
    
    for batch_idx, batch in enumerate(progress_bar):
        # Move to device
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        y1 = batch["primary"].to(device)
        y2 = batch["subcat"].to(device)
        y3 = batch["severity"].to(device)
        
        # Forward pass
        p1, p2, p3 = model(input_ids, attention_mask)
        
        # Calculate losses (same weighting as original)
        loss = criterion(p1, y1) + criterion(p2, y2) + 0.5 * criterion(p3, y3)
        
        # Backward pass with gradient clipping
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        
        # Track accuracy for monitoring
        preds = torch.argmax(p1, dim=1)
        all_predictions.extend(preds.cpu().numpy())
        all_labels.extend(y1.cpu().numpy())
        
        # Update progress bar
        if batch_idx % 10 == 0:
            acc = accuracy_score(all_labels[-batch_size*10:], all_predictions[-batch_size*10:]) if len(all_labels) > 0 else 0
            progress_bar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'avg_loss': f'{total_loss/(batch_idx+1):.4f}',
                'acc': f'{acc:.4f}',
                'lr': f'{scheduler.get_last_lr()[0]:.2e}'
            })
    
    avg_loss = total_loss / len(dataloader)
    return avg_loss

In [14]:
# Cell 14: Evaluation function with detailed metrics
def evaluate(model, dataloader, task_name="Dev"):
    model.eval()
    all_preds = {'primary': [], 'subcat': [], 'severity': []}
    all_true = {'primary': [], 'subcat': [], 'severity': []}
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc=f"Evaluating on {task_name}"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            
            p1, p2, p3 = model(input_ids, attention_mask)
            
            # Primary predictions
            preds1 = torch.argmax(p1, dim=1)
            all_preds['primary'].extend(preds1.cpu().numpy())
            all_true['primary'].extend(batch["primary"].numpy())
            
            # Subcategory predictions
            preds2 = torch.argmax(p2, dim=1)
            all_preds['subcat'].extend(preds2.cpu().numpy())
            all_true['subcat'].extend(batch["subcat"].numpy())
            
            # Severity predictions
            preds3 = torch.argmax(p3, dim=1)
            all_preds['severity'].extend(preds3.cpu().numpy())
            all_true['severity'].extend(batch["severity"].numpy())
    
    # Calculate metrics
    results = {}
    accuracies = {}
    f1_scores = {}
    
    for task in ['primary', 'subcat', 'severity']:
        acc = accuracy_score(all_true[task], all_preds[task])
        f1 = f1_score(all_true[task], all_preds[task], average='weighted')
        accuracies[task] = acc
        f1_scores[task] = f1
        
        results[task] = {
            'accuracy': acc,
            'f1_weighted': f1,
            'predictions': all_preds[task],
            'true_labels': all_true[task]
        }
    
    return results, accuracies, f1_scores

In [15]:
# Cell 15: Main training loop with monitoring
print(f"Starting training at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Device: {device}")
print(f"Total epochs: {num_epochs}")
print("-" * 80)

best_accuracy = 0
best_model_state = None
training_history = []

for epoch in range(num_epochs):
    epoch_start = time.time()
    
    # Training
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, epoch+1, num_epochs)
    
    # Evaluation
    eval_results, accuracies, f1_scores = evaluate(model, dev_loader, "Dev")
    
    # Calculate epoch time
    epoch_time = time.time() - epoch_start
    
    # Save best model
    current_accuracy = accuracies['primary']
    if current_accuracy > best_accuracy:
        best_accuracy = current_accuracy
        best_model_state = model.state_dict().copy()
        torch.save(best_model_state, "models/best_xlmr_multitask.pt")
        print(f"✓ New best model saved! (Accuracy: {best_accuracy:.4f})")
    
    # Record history
    training_history.append({
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'primary_acc': accuracies['primary'],
        'primary_f1': f1_scores['primary'],
        'subcat_acc': accuracies['subcat'],
        'subcat_f1': f1_scores['subcat'],
        'severity_acc': accuracies['severity'],
        'severity_f1': f1_scores['severity'],
        'time': epoch_time
    })
    
    # Print summary
    print(f"\n{'='*60}")
    print(f"Epoch {epoch+1}/{num_epochs} Summary:")
    print(f"{'='*60}")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Dev Primary Accuracy: {accuracies['primary']:.4f} (F1: {f1_scores['primary']:.4f})")
    print(f"  Dev Subcat Accuracy: {accuracies['subcat']:.4f} (F1: {f1_scores['subcat']:.4f})")
    print(f"  Dev Severity Accuracy: {accuracies['severity']:.4f} (F1: {f1_scores['severity']:.4f})")
    print(f"  Time: {epoch_time:.2f}s")
    print(f"{'='*60}\n")
    
    # Clear GPU memory
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\nTraining completed at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Best Accuracy: {best_accuracy:.4f}")
print(f"Best F1: {training_history[training_history.index(max(training_history, key=lambda x: x['primary_acc']))]['primary_f1']:.4f}")

Starting training at 2026-07-24 12:54:29
Device: cuda
Total epochs: 5
--------------------------------------------------------------------------------


Evaluating on Dev: 100%|██████████| 195/195 [00:05<00:00, 33.69it/s]


✓ New best model saved! (Accuracy: 0.8193)

Epoch 1/5 Summary:
  Train Loss: 3.0485
  Dev Primary Accuracy: 0.8193 (F1: 0.8082)
  Dev Subcat Accuracy: 0.7068 (F1: 0.6712)
  Dev Severity Accuracy: 0.6855 (F1: 0.5989)
  Time: 238.61s



Evaluating on Dev: 100%|██████████| 195/195 [00:58<00:00,  3.32it/s]


✓ New best model saved! (Accuracy: 0.8444)

Epoch 2/5 Summary:
  Train Loss: 1.7884
  Dev Primary Accuracy: 0.8444 (F1: 0.8324)
  Dev Subcat Accuracy: 0.7312 (F1: 0.6994)
  Dev Severity Accuracy: 0.7164 (F1: 0.7157)
  Time: 311.17s



Evaluating on Dev: 100%|██████████| 195/195 [00:05<00:00, 33.31it/s]


✓ New best model saved! (Accuracy: 0.8630)

Epoch 3/5 Summary:
  Train Loss: 1.4476
  Dev Primary Accuracy: 0.8630 (F1: 0.8576)
  Dev Subcat Accuracy: 0.7408 (F1: 0.7255)
  Dev Severity Accuracy: 0.7659 (F1: 0.7605)
  Time: 1070.42s



Evaluating on Dev: 100%|██████████| 195/195 [00:11<00:00, 17.55it/s]


✓ New best model saved! (Accuracy: 0.8643)

Epoch 4/5 Summary:
  Train Loss: 1.1918
  Dev Primary Accuracy: 0.8643 (F1: 0.8641)
  Dev Subcat Accuracy: 0.7447 (F1: 0.7329)
  Dev Severity Accuracy: 0.7878 (F1: 0.7886)
  Time: 170.80s



Evaluating on Dev: 100%|██████████| 195/195 [00:05<00:00, 34.03it/s]


✓ New best model saved! (Accuracy: 0.8720)

Epoch 5/5 Summary:
  Train Loss: 1.0173
  Dev Primary Accuracy: 0.8720 (F1: 0.8696)
  Dev Subcat Accuracy: 0.7576 (F1: 0.7495)
  Dev Severity Accuracy: 0.8129 (F1: 0.8103)
  Time: 141.30s


Training completed at 2026-07-24 13:27:07
Best Accuracy: 0.8720
Best F1: 0.8696


In [16]:
# Cell 16: Save final model and encoders
torch.save(model.state_dict(), "models/xlmr_multitask_final.pt")

pickle.dump(le_primary, open("models/le_primary.pkl", "wb"))
pickle.dump(le_subcat, open("models/le_subcat.pkl", "wb"))
pickle.dump(le_severity, open("models/le_severity.pkl", "wb"))

pd.DataFrame(training_history).to_csv("models/training_history.csv", index=False)

print("✓ Models and encoders saved successfully!")

✓ Models and encoders saved successfully!


In [17]:
# Cell 17: Detailed final evaluation
model.eval()
all_preds = {'primary': [], 'subcat': [], 'severity': []}
all_true = {'primary': [], 'subcat': [], 'severity': []}

with torch.no_grad():
    for batch in tqdm(dev_loader, desc="Final Evaluation"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        
        p1, p2, p3 = model(input_ids, attention_mask)
        
        all_preds['primary'].extend(torch.argmax(p1, dim=1).cpu().numpy())
        all_true['primary'].extend(batch["primary"].numpy())
        
        all_preds['subcat'].extend(torch.argmax(p2, dim=1).cpu().numpy())
        all_true['subcat'].extend(batch["subcat"].numpy())
        
        all_preds['severity'].extend(torch.argmax(p3, dim=1).cpu().numpy())
        all_true['severity'].extend(batch["severity"].numpy())

print("="*80)
print("PRIMARY CATEGORY CLASSIFICATION REPORT")
print("="*80)
print(classification_report(all_true['primary'], all_preds['primary'], 
                           target_names=le_primary.classes_))

Final Evaluation: 100%|██████████| 195/195 [00:06<00:00, 31.19it/s]

PRIMARY CATEGORY CLASSIFICATION REPORT
                                     precision    recall  f1-score   support

                        Electricity       0.93      0.89      0.91       193
Encroachment & Illegal Construction       0.48      0.52      0.50        23
                Health & Sanitation       0.64      0.72      0.68        65
                             Others       0.79      0.74      0.76       131
                      Public Safety       0.67      0.12      0.20        17
              Road & Transportation       0.87      0.91      0.89       605
                   Waste Management       0.93      0.91      0.92       487
                      Water Leakage       0.82      0.91      0.86        34

                           accuracy                           0.87      1555
                          macro avg       0.77      0.72      0.72      1555
                       weighted avg       0.87      0.87      0.87      1555



In [18]:
# Cell 18: Prediction function
def predict(text):
    encoding = tokenizer(
        text,
        max_length=128,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    )
    
    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)
    
    with torch.no_grad():
        p_primary, p_subcat, p_severity = model(input_ids, attention_mask)
        
        primary_idx = torch.argmax(p_primary, dim=1).item()
        subcat_idx = torch.argmax(p_subcat, dim=1).item()
        severity_idx = torch.argmax(p_severity, dim=1).item()
    
    return {
        "Primary Category": le_primary.inverse_transform([primary_idx])[0],
        "Sub Category": le_subcat.inverse_transform([subcat_idx])[0],
        "Severity": le_severity.inverse_transform([severity_idx])[0]
    }

In [19]:
# Cell 19: Test predictions
print("="*80)
print("TESTING PREDICTIONS")
print("="*80)

test_texts = [
    "हमारे क्षेत्र की सड़कें बहुत खराब हो चुकी हैं और जगह-जगह गड्ढे हैं, जिससे दुर्घटनाओं का खतरा बढ़ गया है। कृपया जल्द से जल्द मरम्मत करवाई जाए।",
    "Hamare area me baar-baar light chali jati hai, jiski wajah se daily life disturb ho rahi hai. Please proper electricity supply ensure karein.",
    "हमारे क्षेत्र में कचरा नियमित रूप से नहीं उठाया जा रहा है, जिससे गंदगी और बदबू फैल रही है।",
    "There are several potholes in the road, please fix it as soon as possible."
]

for i, text in enumerate(test_texts, 1):
    print(f"\nTest {i}:")
    print(f"Text: {text[:100]}...")
    result = predict(text)
    print(f"  Primary: {result['Primary Category']}")
    print(f"  Sub: {result['Sub Category']}")
    print(f"  Severity: {result['Severity']}")

TESTING PREDICTIONS

Test 1:
Text: हमारे क्षेत्र की सड़कें बहुत खराब हो चुकी हैं और जगह-जगह गड्ढे हैं, जिससे दुर्घटनाओं का खतरा बढ़ गया...
  Primary: Road & Transportation
  Sub: Road Damage & Potholes
  Severity: MEDIUM

Test 2:
Text: Hamare area me baar-baar light chali jati hai, jiski wajah se daily life disturb ho rahi hai. Please...
  Primary: Electricity
  Sub: Streetlights & Public Lighting
  Severity: LOW

Test 3:
Text: हमारे क्षेत्र में कचरा नियमित रूप से नहीं उठाया जा रहा है, जिससे गंदगी और बदबू फैल रही है।...
  Primary: Waste Management
  Sub: Garbage Dumping & Black Spots
  Severity: LOW

Test 4:
Text: There are several potholes in the road, please fix it as soon as possible....
  Primary: Road & Transportation
  Sub: Road Damage & Potholes
  Severity: LOW


In [20]:
# # Cell 20: Prediction function with confidence scores
# def predict(text, return_probs=False):
#     """Predict categories for a given text"""
#     encoding = tokenizer(
#         text,
#         max_length=128,
#         truncation=True,
#         padding="max_length",
#         return_tensors="pt"
#     )
    
#     input_ids = encoding["input_ids"].to(device)
#     attention_mask = encoding["attention_mask"].to(device)
    
#     with torch.no_grad():
#         p_primary, p_subcat, p_severity = model(input_ids, attention_mask)
        
#         # Get probabilities
#         probs_primary = F.softmax(p_primary, dim=1)
#         probs_subcat = F.softmax(p_subcat, dim=1)
#         probs_severity = F.softmax(p_severity, dim=1)
        
#         # Get predictions
#         primary_idx = torch.argmax(p_primary, dim=1).item()
#         subcat_idx = torch.argmax(p_subcat, dim=1).item()
#         severity_idx = torch.argmax(p_severity, dim=1).item()
    
#     result = {
#         "Primary Category": le_primary.inverse_transform([primary_idx])[0],
#         "Primary Confidence": probs_primary[0][primary_idx].item(),
#         "Sub Category": le_subcat.inverse_transform([subcat_idx])[0],
#         "Sub Confidence": probs_subcat[0][subcat_idx].item(),
#         "Severity": le_severity.inverse_transform([severity_idx])[0],
#         "Severity Confidence": probs_severity[0][severity_idx].item()
#     }
    
#     if return_probs:
#         result["Primary Probabilities"] = {le_primary.classes_[i]: probs_primary[0][i].item() 
#                                            for i in range(len(le_primary.classes_))}
    
#     return result

# def predict_batch(texts):
#     """Predict categories for multiple texts"""
#     return [predict(text) for text in texts]

In [22]:
predict("हमारे क्षेत्र की सड़कें बहुत खराब हो चुकी हैं और जगह-जगह गड्ढे हैं, जिससे दुर्घटनाओं का खतरा बढ़ गया है। कृपया जल्द से जल्द मरम्मत करवाई जाए।")

{'Primary Category': 'Road & Transportation',
 'Sub Category': 'Road Damage & Potholes',
 'Severity': 'MEDIUM'}

In [23]:
predict("Hamare area me baar-baar light chali jati hai, jiski wajah se daily life disturb ho rahi hai. Please proper electricity supply ensure karein.")

{'Primary Category': 'Electricity',
 'Sub Category': 'Streetlights & Public Lighting',
 'Severity': 'LOW'}

In [24]:
predict("हमारे क्षेत्र में कचरा नियमित रूप से नहीं उठाया जा रहा है, जिससे गंदगी और बदबू फैल रही है।")

{'Primary Category': 'Waste Management',
 'Sub Category': 'Garbage Dumping & Black Spots',
 'Severity': 'LOW'}

In [25]:
predict("There are several patholes in road please fix it as soons as possible")

{'Primary Category': 'Road & Transportation',
 'Sub Category': 'Road Damage & Potholes',
 'Severity': 'LOW'}